# 日内早盘动量稳定超额研究

只用 qlib 原生 15 分钟 provider，检验开盘 60 分钟可观测路径能否映射为 10:30–收盘的
SOXL/SOXX/现金仓位并稳定超越 SOXX。无收益预测、无拟合参数、无隔夜仓。


## 1. 预登记计划


In [1]:
import sys, hashlib, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
CARD = ROOT / 'research' / 'intraday_early_momentum_20261010'
plan_bytes = (CARD / 'plan.json').read_bytes()
plan = json.loads(plan_bytes)
print('plan sha256:', hashlib.sha256(plan_bytes).hexdigest())
print('provider:', plan['provider_uri'], '| as_of:', plan['as_of'])
print('segments:', plan['segments'])


plan sha256: b26ef641e60a632990e43c79d7d22ae787afd86989bcbac86f8e943eeb28dce5
provider: data/qlib/soxl_intraday_15min | as_of: 2026-10-07
segments: {'development': ['2020-01-02', '2023-12-29'], 'reused_diagnostic': ['2024-01-02', '2026-06-30'], 'forward': ['2026-07-01', '2026-10-07']}


## 2. 运行（qlib 原生 + Recorder）


In [2]:
from src.workflow import intraday_early_momentum as workflow
from src.workflow.intraday_early_momentum import write_card_files

summary = workflow.run_intraday_early_momentum(root=ROOT)
write_card_files(summary, card_dir=CARD)
print('decision:', summary['decision'])
print('selected policy:', summary['selected_policy'])
print('promotion_eligible:', summary['promotion_eligible'])
print('recorder_id:', summary['recorder_id'])
print('provider_uri:', summary['provider_uri'])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[38753:MainThread](2026-10-09 16:42:46,826) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/09 16:42:47 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[38753:MainThread](2026-10-09 16:42:47,203) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[38753:MainThread](2026-10-09 16:42:47,203) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min')}


...

[38753:MainThread](2026-10-09 16:42:47,822) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name intraday_early_momentum_20261010.


[38753:MainThread](2026-10-09 16:42:47,823) INFO - qlib.workflow - [exp.py:258] - Experiment 878558123369358088 starts running ...


[38753:MainThread](2026-10-09 16:42:47,876) INFO - qlib.workflow - [recorder.py:345] - Recorder b0e963a5695c40f49bb64934d820df25 starts running under Experiment 878558123369358088 ...


[38753:MainThread](2026-10-09 16:42:47,959) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


decision: no_stable_intraday_excess_in_registered_family
selected policy: soxx_momentum_l20
promotion_eligible: False
recorder_id: b0e963a5695c40f49bb64934d820df25
provider_uri: /Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min


## 3. 候选对比（10bp）


In [3]:
import pandas as pd
comp = pd.read_csv(CARD / 'policy_comparison.csv')
cols = ['window','policy','return_pct','benchmark_return_pct','max_drawdown_pct',
        'benchmark_max_drawdown_pct','sharpe','benchmark_sharpe','sharpe_difference',
        'stable_excess','positive_year_count','year_count','total_turnover']
display(comp[cols].round(3))


,window,policy,return_pct,benchmark_return_pct,max_drawdown_pct,benchmark_max_drawdown_pct,sharpe,benchmark_sharpe,sharpe_difference,stable_excess,positive_year_count,year_count,total_turnover
0,development,early_momentum_off,26.563,26.563,-32.644,-32.644,0.372,0.372,0.000,False,0,4,0.0
1,development,soxl_momentum_l5,-38.028,26.563,-74.025,-32.644,-0.003,0.372,-0.375,False,2,4,988.0
2,development,soxl_momentum_l10,-34.519,26.563,-68.354,-32.644,0.001,0.372,-0.371,False,1,4,920.0
3,development,soxl_momentum_l20,33.133,26.563,-50.396,-32.644,0.381,0.372,0.009,False,3,4,716.0
4,development,soxx_momentum_l5,-25.959,26.563,-64.546,-32.644,0.055,0.372,-0.317,False,1,4,804.0
5,development,soxx_momentum_l10,25.758,26.563,-47.022,-32.644,0.341,0.372,-0.031,False,2,4,516.0
6,development,soxx_momentum_l20,32.573,26.563,-31.351,-32.644,0.395,0.372,0.022,True,2,4,100.0
7,reused_diagnostic,early_momentum_off,10.737,10.737,-35.117,-35.117,0.291,0.291,0.000,False,0,3,0.0
8,reused_diagnostic,soxl_momentum_l5,0.125,10.737,-57.899,-35.117,0.240,0.291,-0.051,False,1,3,610.0
9,reused_diagnostic,soxl_momentum_l10,-0.669,10.737,-59.772,-35.117,0.225,0.291,-0.066,False,1,3,570.0


## 4. 选中策略年度分解


In [4]:
display(pd.read_csv(CARD / 'annual_metrics.csv').round(4))
print('forward verdict:', json.dumps(summary['selected_forward_verdict'], indent=2))


,window,policy,year,sessions,wealth,average_soxl_weight,average_cash_weight,turnover,sessions_bench,wealth_bench,average_soxl_weight_bench,average_cash_weight_bench,turnover_bench,wealth_ratio_vs_soxx
0,development,soxx_momentum_l20,2020,251,0.8385,0.0199,0.0,20.0,251,0.8296,0.0,0.0,0.0,1.0106
1,development,soxx_momentum_l20,2021,251,1.1061,0.0080,0.0,8.0,251,1.1237,0.0,0.0,0.0,0.9843
2,development,soxx_momentum_l20,2022,250,0.9181,0.0600,0.0,56.0,250,0.9446,0.0,0.0,0.0,0.9719
3,development,soxx_momentum_l20,2023,248,1.5569,0.0161,0.0,16.0,248,1.4371,0.0,0.0,0.0,1.0834
4,reused_diagnostic,soxx_momentum_l20,2024,249,0.8780,0.0040,0.0,4.0,249,0.8511,0.0,0.0,0.0,1.0315
5,reused_diagnostic,soxx_momentum_l20,2025,247,0.9638,0.0243,0.0,24.0,247,0.9596,0.0,0.0,0.0,1.0043
6,reused_diagnostic,soxx_momentum_l20,2026,123,1.3526,0.0407,0.0,14.0,123,1.3558,0.0,0.0,0.0,0.9976
7,forward,soxx_momentum_l20,2026,67,0.9210,0.0896,0.0,24.0,67,0.9279,0.0,0.0,0.0,0.9926


forward verdict: {
  "beats_return_and_drawdown": false,
  "beats_sharpe": true,
  "positive_year_count": 0,
  "year_count": 1,
  "minimum_positive_years": 2,
  "annual_win_rate": 0.0,
  "stable_excess": false,
  "reference": "SOXX buy-and-hold, identical sessions and cost model",
  "scope": "reused_history_and_forward_diagnostic_not_production_evidence"
}


## 5. 中文结论


In [5]:
o = summary['selected_outer']['strategy']; f = summary['selected_forward']['strategy']
print(f"选中 {summary['selected_policy']}；候选族 {summary['candidate_count']} 条；无收益预测、无隔夜仓。")
print(f"复用区间 {summary['segments']['reused_diagnostic']}：收益 {o['return_pct']:.2f}%，回撤 {o['max_drawdown_pct']:.2f}%，Sharpe {o['sharpe']:.2f}。")
print(f"前向区间 {summary['segments']['forward']}：收益 {f['return_pct']:.2f}%，回撤 {f['max_drawdown_pct']:.2f}%，Sharpe {f['sharpe']:.2f}。")
print('是否晋升:', summary['promotion_eligible'], '| 决定:', summary['decision'])
print('qlib Recorder:', summary['recorder_id'])


选中 soxx_momentum_l20；候选族 7 条；无收益预测、无隔夜仓。
复用区间 ['2024-01-02', '2026-06-30']：收益 14.45%，回撤 -32.26%，Sharpe 0.34。
前向区间 ['2026-07-01', '2026-10-07']：收益 -7.90%，回撤 -14.47%，Sharpe -1.30。
是否晋升: False | 决定: no_stable_intraday_excess_in_registered_family
qlib Recorder: b0e963a5695c40f49bb64934d820df25
